In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
import os
import glob

from sklearn.preprocessing import (
    LabelEncoder,
    StandardScaler,
    MinMaxScaler,
)

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from typing import Dict
from sklearn.model_selection import cross_val_score, KFold
from sklearn.model_selection import train_test_split
from sklearn.model_selection import TimeSeriesSplit
from sklearn.utils.class_weight import compute_class_weight
import warnings

     
from sklearn.metrics import (
    roc_auc_score, 
    roc_curve, 
    confusion_matrix, 
    precision_score, 
    recall_score, 
    accuracy_score,
    classification_report
)

warnings.filterwarnings('ignore')


# import warnings
# warnings.filterwarnings('ignore')


In [2]:
machine_number = 1
hours= 24

In [3]:
machine = pd.read_csv(f"../../../data/azure_pm/machines/machine_{machine_number}.csv")
machine_lag = pd.read_csv(f"../../../data/azure_pm/lag_features/machine_{machine_number}_lag_features.csv")

In [5]:
def rows_n_hours_before_failure(machine, machine_failure, hours):  
    failure_times = machine_failure["datetime"]

    # Convert the datetime columns to datetime if they're not already
    machine["datetime"] = pd.to_datetime(machine["datetime"])
    failure_times = pd.to_datetime(failure_times)

    # Initialize an empty list to store the rows and their indices
    rows = []
    indices = []

    # Iterate over each failure time and its index
    for idx_failure, failure_time in zip(machine_failure.index, failure_times):
        # Calculate the time n hours before the failure
        target_time = failure_time - pd.Timedelta(hours=hours)
        
        # Get the row with the closest time to the target time
        idx = (machine["datetime"] - target_time).abs().idxmin()
        closest_row = machine.loc[idx]
        
        # Append the row and the failure index to the lists
        rows.append(closest_row)
        indices.append(idx_failure)

    # Create a new dataframe with the rows
    machine_1_prev_24h = pd.DataFrame(rows)
    machine_1_prev_24h["id_failure_row"] = indices

    return machine_1_prev_24h

In [6]:
cols = ["datetime", "failure", "id_failure_row"]
machine_failure = (machine[machine['failure'] != '0'])
machine_24_before_failure = (rows_n_hours_before_failure(machine=machine, machine_failure=machine[machine['failure'] != '0'], hours=hours))
lag_failure = (rows_n_hours_before_failure(machine=machine_lag, machine_failure=machine_lag[machine_lag['failure'] != 0], hours=hours))

display(machine_failure)
display(machine_24_before_failure)
display(lag_failure)

,datetime,machineID,volt,rotate,pressure,vibration,model,age,errorID,comp,failure
96,2015-01-05 06:00:00,1,179.303153,499.777962,111.833028,52.383097,model3,18,0,comp4,comp4
97,2015-01-05 06:00:00,1,179.303153,499.777962,111.833028,52.383097,model3,18,0,comp1,comp4
1539,2015-03-06 06:00:00,1,198.257975,456.862342,89.333995,38.671900,model3,18,0,comp1,comp1
2620,2015-04-20 06:00:00,1,180.050801,346.362480,105.661164,39.218055,model3,18,0,comp2,comp2
4061,2015-06-19 06:00:00,1,187.673963,493.005160,105.334392,53.963961,model3,18,0,comp1,comp4
4062,2015-06-19 06:00:00,1,187.673963,493.005160,105.334392,53.963961,model3,18,0,comp4,comp4
5862,2015-09-02 06:00:00,1,144.094532,409.380150,106.720871,57.454990,model3,18,0,comp1,comp4
5863,2015-09-02 06:00:00,1,144.094532,409.380150,106.720871,57.454990,model3,18,0,comp4,comp4
6945,2015-10-17 06:00:00,1,178.322428,383.715256,79.704008,43.213417,model3,18,0,comp4,comp2
6946,2015-10-17 06:00:00,1,178.322428,383.715256,79.704008,43.213417,model3,18,0,comp2,comp2


,datetime,machineID,volt,rotate,pressure,vibration,model,age,errorID,comp,failure,id_failure_row
72,2015-01-04 06:00:00,1,165.010140,448.468838,97.709630,48.238941,model3,18,error5,0,0,96
72,2015-01-04 06:00:00,1,165.010140,448.468838,97.709630,48.238941,model3,18,error5,0,0,97
1515,2015-03-05 06:00:00,1,208.701202,383.992489,103.683080,42.557824,model3,18,error1,0,0,1539
2595,2015-04-19 06:00:00,1,156.765379,380.849889,94.354998,37.436059,model3,18,error2,0,0,2620
4037,2015-06-18 06:00:00,1,167.428048,443.082952,96.968145,51.105583,model3,18,error5,0,0,4061
4037,2015-06-18 06:00:00,1,167.428048,443.082952,96.968145,51.105583,model3,18,error5,0,0,4062
5838,2015-09-01 06:00:00,1,188.464713,433.273533,121.728619,46.192763,model3,18,error5,0,0,5862
5838,2015-09-01 06:00:00,1,188.464713,433.273533,121.728619,46.192763,model3,18,error5,0,0,5863
6920,2015-10-16 06:00:00,1,203.130813,304.774518,101.433608,39.833376,model3,18,error2,0,0,6945
6920,2015-10-16 06:00:00,1,203.130813,304.774518,101.433608,39.833376,model3,18,error2,0,0,6946


,datetime,volt,rotate,pressure,vibration,errorID,comp,failure,target,volt_lag_1h,...,error_count_24h,maint_count_6h,maint_count_24h,hour,day_of_week,is_weekend,is_working_hours,hours_since_maint,hours_since_error,id_failure_row
72,2015-01-04 06:00:00,0.386535,0.553421,0.403796,0.568691,5,0,0,3,0.433966,...,9.0,0.0,0.0,6,6,1,0,72,0,96
72,2015-01-04 06:00:00,0.386535,0.553421,0.403796,0.568691,5,0,0,3,0.433966,...,9.0,0.0,0.0,6,6,1,0,72,0,97
1515,2015-03-05 06:00:00,0.754059,0.400178,0.465682,0.442350,1,0,0,1,0.606900,...,1.0,0.0,0.0,6,3,0,0,336,0,1539
2595,2015-04-19 06:00:00,0.317181,0.392709,0.369041,0.328448,2,0,0,0,0.476150,...,2.0,0.0,0.0,6,6,1,0,336,0,2620
4037,2015-06-18 06:00:00,0.406874,0.540621,0.396114,0.632441,5,0,0,3,0.535453,...,5.0,0.0,0.0,6,3,0,0,336,0,4061
4037,2015-06-18 06:00:00,0.406874,0.540621,0.396114,0.632441,5,0,0,3,0.535453,...,5.0,0.0,0.0,6,3,0,0,336,0,4062
5838,2015-09-01 06:00:00,0.583832,0.517306,0.652636,0.523186,5,0,0,3,0.543068,...,5.0,0.0,0.0,6,1,0,0,696,0,5862
5838,2015-09-01 06:00:00,0.583832,0.517306,0.652636,0.523186,5,0,0,3,0.543068,...,5.0,0.0,0.0,6,1,0,0,696,0,5863
6920,2015-10-16 06:00:00,0.707202,0.211898,0.442377,0.381762,3,0,0,0,0.410665,...,6.0,0.0,0.0,6,4,0,0,336,0,6945
6920,2015-10-16 06:00:00,0.707202,0.211898,0.442377,0.381762,3,0,0,0,0.410665,...,6.0,0.0,0.0,6,4,0,0,336,0,6946


In [10]:
def update_lag_failure_target(machine_failure, machine_24_before_failure, lag_failure):
    """
    Updates the 'target' variable in lag_failure with the corresponding 'failure' value from machine_failure,
    using machine_24_before_failure as a bridge for the join.
    """
    if 'id_failure_row' not in machine_24_before_failure.columns:
        raise ValueError("machine_24_before_failure must have 'id_failure_row' column.")

    failure_values = []
    for idx in lag_failure.index:
        if idx in machine_24_before_failure.index:
            id_failure_row = machine_24_before_failure.loc[idx, 'id_failure_row']
            # If multiple rows, take the first
            if isinstance(id_failure_row, pd.Series):
                id_failure_row = id_failure_row.iloc[0]
            failure_val = machine_failure.loc[id_failure_row, 'failure'] if id_failure_row in machine_failure.index else None
        else:
            failure_val = None
        failure_values.append(failure_val)

    lag_failure = lag_failure.copy()
    lag_failure['target'] = failure_values
    return lag_failure

machine_failure = (machine[machine['failure'] != '0'])
machine_24_before_failure = (rows_n_hours_before_failure(machine=machine, machine_failure=machine[machine['failure'] != '0'], hours=hours))
lag_failure = (rows_n_hours_before_failure(machine=machine_lag, machine_failure=machine_lag[machine_lag['failure'] != 0], hours=hours))
lag_failure_updated = update_lag_failure_target(machine_failure, machine_24_before_failure, lag_failure)


In [ ]:
# cols = ["datetime", "failure", "id_failure_row"]
# machine_failure = (machine[machine['failure'] != '0'])
# machine_24_before_failure = (rows_n_hours_before_failure(machine=machine, machine_failure=machine[machine['failure'] != '0'], hours=hours))
# lag_failure = (rows_n_hours_before_failure(machine=machine_lag, machine_failure=machine_lag[machine_lag['failure'] != 0], hours=hours))

display(machine_failure)
# display(machine_24_before_failure)
# display(lag_failure)
display(lag_failure_updated)


,datetime,machineID,volt,rotate,pressure,vibration,model,age,errorID,comp,failure
96,2015-01-05 06:00:00,1,179.303153,499.777962,111.833028,52.383097,model3,18,0,comp4,comp4
97,2015-01-05 06:00:00,1,179.303153,499.777962,111.833028,52.383097,model3,18,0,comp1,comp4
1539,2015-03-06 06:00:00,1,198.257975,456.862342,89.333995,38.671900,model3,18,0,comp1,comp1
2620,2015-04-20 06:00:00,1,180.050801,346.362480,105.661164,39.218055,model3,18,0,comp2,comp2
4061,2015-06-19 06:00:00,1,187.673963,493.005160,105.334392,53.963961,model3,18,0,comp1,comp4
4062,2015-06-19 06:00:00,1,187.673963,493.005160,105.334392,53.963961,model3,18,0,comp4,comp4
5862,2015-09-02 06:00:00,1,144.094532,409.380150,106.720871,57.454990,model3,18,0,comp1,comp4
5863,2015-09-02 06:00:00,1,144.094532,409.380150,106.720871,57.454990,model3,18,0,comp4,comp4
6945,2015-10-17 06:00:00,1,178.322428,383.715256,79.704008,43.213417,model3,18,0,comp4,comp2
6946,2015-10-17 06:00:00,1,178.322428,383.715256,79.704008,43.213417,model3,18,0,comp2,comp2


,datetime,volt,rotate,pressure,vibration,errorID,comp,failure,target,volt_lag_1h,...,error_count_24h,maint_count_6h,maint_count_24h,hour,day_of_week,is_weekend,is_working_hours,hours_since_maint,hours_since_error,id_failure_row
72,2015-01-04 06:00:00,0.386535,0.553421,0.403796,0.568691,5,0,0,comp4,0.433966,...,9.0,0.0,0.0,6,6,1,0,72,0,96
72,2015-01-04 06:00:00,0.386535,0.553421,0.403796,0.568691,5,0,0,comp4,0.433966,...,9.0,0.0,0.0,6,6,1,0,72,0,97
1515,2015-03-05 06:00:00,0.754059,0.400178,0.465682,0.442350,1,0,0,comp1,0.606900,...,1.0,0.0,0.0,6,3,0,0,336,0,1539
2595,2015-04-19 06:00:00,0.317181,0.392709,0.369041,0.328448,2,0,0,comp2,0.476150,...,2.0,0.0,0.0,6,6,1,0,336,0,2620
4037,2015-06-18 06:00:00,0.406874,0.540621,0.396114,0.632441,5,0,0,comp4,0.535453,...,5.0,0.0,0.0,6,3,0,0,336,0,4061
4037,2015-06-18 06:00:00,0.406874,0.540621,0.396114,0.632441,5,0,0,comp4,0.535453,...,5.0,0.0,0.0,6,3,0,0,336,0,4062
5838,2015-09-01 06:00:00,0.583832,0.517306,0.652636,0.523186,5,0,0,comp4,0.543068,...,5.0,0.0,0.0,6,1,0,0,696,0,5862
5838,2015-09-01 06:00:00,0.583832,0.517306,0.652636,0.523186,5,0,0,comp4,0.543068,...,5.0,0.0,0.0,6,1,0,0,696,0,5863
6920,2015-10-16 06:00:00,0.707202,0.211898,0.442377,0.381762,3,0,0,comp2,0.410665,...,6.0,0.0,0.0,6,4,0,0,336,0,6945
6920,2015-10-16 06:00:00,0.707202,0.211898,0.442377,0.381762,3,0,0,comp2,0.410665,...,6.0,0.0,0.0,6,4,0,0,336,0,6946


In [30]:
display(machine_failure[["datetime", "failure"]])
display(lag_failure_updated[["datetime", "failure", "target"]])

,datetime,failure
96,2015-01-05 06:00:00,comp4
97,2015-01-05 06:00:00,comp4
1539,2015-03-06 06:00:00,comp1
2620,2015-04-20 06:00:00,comp2
4061,2015-06-19 06:00:00,comp4
4062,2015-06-19 06:00:00,comp4
5862,2015-09-02 06:00:00,comp4
5863,2015-09-02 06:00:00,comp4
6945,2015-10-17 06:00:00,comp2
6946,2015-10-17 06:00:00,comp2


,datetime,failure,target
72,2015-01-04 06:00:00,0,comp4
72,2015-01-04 06:00:00,0,comp4
1515,2015-03-05 06:00:00,0,comp1
2595,2015-04-19 06:00:00,0,comp2
4037,2015-06-18 06:00:00,0,comp4
4037,2015-06-18 06:00:00,0,comp4
5838,2015-09-01 06:00:00,0,comp4
5838,2015-09-01 06:00:00,0,comp4
6920,2015-10-16 06:00:00,0,comp2
6920,2015-10-16 06:00:00,0,comp2


In [25]:
t1 = display(machine_failure["datetime"][:1])
t_1 = display(lag_failure_updated["datetime"][:1])

96   2015-01-05 06:00:00
Name: datetime, dtype: datetime64[ns]

72   2015-01-04 06:00:00
Name: datetime, dtype: datetime64[ns]

In [31]:
lag_failure

,datetime,volt,rotate,pressure,vibration,errorID,comp,failure,target,volt_lag_1h,...,error_count_24h,maint_count_6h,maint_count_24h,hour,day_of_week,is_weekend,is_working_hours,hours_since_maint,hours_since_error,id_failure_row
72,2015-01-04 06:00:00,0.386535,0.553421,0.403796,0.568691,5,0,0,3,0.433966,...,9.0,0.0,0.0,6,6,1,0,72,0,96
72,2015-01-04 06:00:00,0.386535,0.553421,0.403796,0.568691,5,0,0,3,0.433966,...,9.0,0.0,0.0,6,6,1,0,72,0,97
1515,2015-03-05 06:00:00,0.754059,0.400178,0.465682,0.442350,1,0,0,1,0.606900,...,1.0,0.0,0.0,6,3,0,0,336,0,1539
2595,2015-04-19 06:00:00,0.317181,0.392709,0.369041,0.328448,2,0,0,0,0.476150,...,2.0,0.0,0.0,6,6,1,0,336,0,2620
4037,2015-06-18 06:00:00,0.406874,0.540621,0.396114,0.632441,5,0,0,3,0.535453,...,5.0,0.0,0.0,6,3,0,0,336,0,4061
4037,2015-06-18 06:00:00,0.406874,0.540621,0.396114,0.632441,5,0,0,3,0.535453,...,5.0,0.0,0.0,6,3,0,0,336,0,4062
5838,2015-09-01 06:00:00,0.583832,0.517306,0.652636,0.523186,5,0,0,3,0.543068,...,5.0,0.0,0.0,6,1,0,0,696,0,5862
5838,2015-09-01 06:00:00,0.583832,0.517306,0.652636,0.523186,5,0,0,3,0.543068,...,5.0,0.0,0.0,6,1,0,0,696,0,5863
6920,2015-10-16 06:00:00,0.707202,0.211898,0.442377,0.381762,3,0,0,0,0.410665,...,6.0,0.0,0.0,6,4,0,0,336,0,6945
6920,2015-10-16 06:00:00,0.707202,0.211898,0.442377,0.381762,3,0,0,0,0.410665,...,6.0,0.0,0.0,6,4,0,0,336,0,6946


In [37]:
import pandas as pd
import numpy as np

def create_safe_non_failure_samples_enhanced(machine_lag, lag_failure_updated, hours_before=24, safe_buffer_hours=48):
    """
    Create non-failure samples from 'safe zones' using the full feature set from machine_lag
    
    Parameters:
    - machine_lag: Full dataset with all engineered features
    - lag_failure_updated: Your target dataset (24h before failures)
    - hours_before: Lead time before failure (24h)
    - safe_buffer_hours: Additional buffer to ensure truly safe periods (48h recommended)
    """
    
    # Convert datetime columns to datetime if they aren't already
    machine_lag['datetime'] = pd.to_datetime(machine_lag['datetime'])
    lag_failure_updated['datetime'] = pd.to_datetime(lag_failure_updated['datetime'])
    
    # Get all failure timestamps from machine_lag where failure = 1
    failure_timestamps = machine_lag[machine_lag['failure'] == 1]['datetime']
    
    # Create exclusion zones around each failure
    exclusion_periods = []
    
    for failure_time in failure_timestamps:
        # Exclude from (failure_time - safe_buffer_hours) to failure_time
        start_exclusion = failure_time - pd.Timedelta(hours=safe_buffer_hours)
        end_exclusion = failure_time
        exclusion_periods.append((start_exclusion, end_exclusion))
    
    # Also exclude the timestamps that are already in lag_failure_updated (24h before failures)
    existing_target_timestamps = set(lag_failure_updated['datetime'])
    
    # Filter machine_lag to find safe timestamps
    safe_candidates = machine_lag.copy()
    
    # Remove rows that fall in exclusion zones
    def is_in_exclusion_zone(timestamp):
        for start_excl, end_excl in exclusion_periods:
            if start_excl <= timestamp <= end_excl:
                return True
        return False
    
    # Filter out exclusion zones and existing target timestamps
    safe_mask = (
        ~safe_candidates['datetime'].apply(is_in_exclusion_zone) &
        ~safe_candidates['datetime'].isin(existing_target_timestamps) &
        (safe_candidates['failure'] == 0)  # Only non-failure records
    )
    
    safe_candidates = safe_candidates[safe_mask]
    
    if len(safe_candidates) == 0:
        print("Warning: No safe candidates found. Consider reducing safe_buffer_hours.")
        return pd.DataFrame()
    
    # Sample non-failure records
    n_samples = min(len(lag_failure_updated), len(safe_candidates))
    
    if n_samples > len(safe_candidates):
        print(f"Warning: Only {len(safe_candidates)} safe candidates available, but {len(lag_failure_updated)} samples requested.")
        n_samples = len(safe_candidates)
    
    # Randomly sample from safe candidates
    non_failure_samples = safe_candidates.sample(n=n_samples, random_state=42).copy()
    
    # Set target to 0 for non-failure samples (they should already be 0, but ensure it)
    non_failure_samples['target'] = 0
    
    # Reset index
    non_failure_samples = non_failure_samples.reset_index(drop=True)
    
    return non_failure_samples

def create_balanced_dataset(machine_lag, lag_failure_updated, hours_before=24, safe_buffer_hours=48):
    """
    Create a balanced dataset combining failure predictions and safe non-failure samples
    """
    
    # Get non-failure samples
    non_failure_df = create_safe_non_failure_samples_enhanced(
        machine_lag, lag_failure_updated, hours_before, safe_buffer_hours
    )
    
    if len(non_failure_df) == 0:
        return lag_failure_updated.copy()
    
    # Ensure both dataframes have the same columns
    common_columns = list(set(lag_failure_updated.columns) & set(non_failure_df.columns))
    
    # If lag_failure_updated is missing some features, we need to merge them from machine_lag
    if len(common_columns) < len(machine_lag.columns):
        print("Merging additional features from machine_lag to lag_failure_updated...")
        
        # Merge lag_failure_updated with machine_lag to get all features
        lag_failure_enhanced = pd.merge(
            lag_failure_updated, 
            machine_lag, 
            on='datetime', 
            how='left',
            suffixes=('', '_from_machine_lag')
        )
        
        # Clean up duplicate columns (keep the original values from lag_failure_updated)
        for col in lag_failure_enhanced.columns:
            if col.endswith('_from_machine_lag'):
                original_col = col.replace('_from_machine_lag', '')
                if original_col in lag_failure_enhanced.columns:
                    lag_failure_enhanced[original_col] = lag_failure_enhanced[original_col].fillna(
                        lag_failure_enhanced[col]
                    )
                    lag_failure_enhanced = lag_failure_enhanced.drop(columns=[col])
        
        lag_failure_to_use = lag_failure_enhanced
    else:
        lag_failure_to_use = lag_failure_updated.copy()
    
    # Ensure all required columns are present in both dataframes
    all_required_columns = list(machine_lag.columns)
    
    # Add missing columns with appropriate default values if needed
    for col in all_required_columns:
        if col not in lag_failure_to_use.columns:
            lag_failure_to_use[col] = np.nan
        if col not in non_failure_df.columns:
            non_failure_df[col] = np.nan
    
    # Select only the required columns in the same order
    lag_failure_to_use = lag_failure_to_use[all_required_columns]
    non_failure_df = non_failure_df[all_required_columns]
    
    # Combine datasets
    balanced_dataset = pd.concat([lag_failure_to_use, non_failure_df], ignore_index=True)
    balanced_dataset = balanced_dataset.sort_values('datetime').reset_index(drop=True)
    
    return balanced_dataset, non_failure_df

# Usage example:
"""
# Create the balanced dataset
balanced_dataset, non_failure_df = create_balanced_dataset(machine_lag, lag_failure_updated)

print(f"Original failure samples (target=1): {len(lag_failure_updated)}")
print(f"New non-failure samples (target=0): {len(non_failure_df)}")
print(f"Total balanced dataset: {len(balanced_dataset)}")
print(f"Dataset columns: {len(balanced_dataset.columns)}")
print(f"Target distribution:")
print(balanced_dataset['target'].value_counts())
"""

# Alternative: If you want more control over sampling strategy
def create_stratified_non_failure_samples(machine_lag, lag_failure_updated, 
                                         safe_buffer_hours=48, 
                                         stratify_by=['comp', 'is_weekend', 'is_working_hours']):
    """
    Create stratified non-failure samples to ensure diversity across different conditions
    """
    
    machine_lag['datetime'] = pd.to_datetime(machine_lag['datetime'])
    lag_failure_updated['datetime'] = pd.to_datetime(lag_failure_updated['datetime'])
    
    # Get failure timestamps and create exclusion zones
    failure_timestamps = machine_lag[machine_lag['failure'] == 1]['datetime']
    exclusion_periods = []
    
    for failure_time in failure_timestamps:
        start_exclusion = failure_time - pd.Timedelta(hours=safe_buffer_hours)
        end_exclusion = failure_time
        exclusion_periods.append((start_exclusion, end_exclusion))
    
    # Filter safe candidates
    def is_in_exclusion_zone(timestamp):
        for start_excl, end_excl in exclusion_periods:
            if start_excl <= timestamp <= end_excl:
                return True
        return False
    
    existing_target_timestamps = set(lag_failure_updated['datetime'])
    
    safe_mask = (
        ~machine_lag['datetime'].apply(is_in_exclusion_zone) &
        ~machine_lag['datetime'].isin(existing_target_timestamps) &
        (machine_lag['failure'] == 0)
    )
    
    safe_candidates = machine_lag[safe_mask].copy()
    
    if len(safe_candidates) == 0:
        return pd.DataFrame()
    
    # Stratified sampling
    target_samples = len(lag_failure_updated)
    
    # Get the distribution of stratification variables in lag_failure_updated
    if all(col in lag_failure_updated.columns for col in stratify_by):
        # Sample proportionally to match the failure sample distribution
        stratified_samples = []
        
        for group_values, group_df in lag_failure_updated.groupby(stratify_by):
            group_size = len(group_df)
            proportion = group_size / len(lag_failure_updated)
            target_group_size = int(proportion * target_samples)
            
            # Find matching safe candidates
            mask = pd.Series(True, index=safe_candidates.index)
            for i, col in enumerate(stratify_by):
                mask &= (safe_candidates[col] == group_values[i])
            
            group_safe_candidates = safe_candidates[mask]
            
            if len(group_safe_candidates) > 0:
                sample_size = min(target_group_size, len(group_safe_candidates))
                group_sample = group_safe_candidates.sample(n=sample_size, random_state=42)
                stratified_samples.append(group_sample)
        
        if stratified_samples:
            non_failure_samples = pd.concat(stratified_samples, ignore_index=True)
        else:
            # Fallback to random sampling
            non_failure_samples = safe_candidates.sample(
                n=min(target_samples, len(safe_candidates)), 
                random_state=42
            )
    else:
        # Fallback to random sampling if stratification columns not available
        non_failure_samples = safe_candidates.sample(
            n=min(target_samples, len(safe_candidates)), 
            random_state=42
        )
    
    non_failure_samples['target'] = 0
    return non_failure_samples.reset_index(drop=True)

In [38]:
# Create the balanced dataset with all features
balanced_dataset, non_failure_df = create_balanced_dataset(machine_lag, lag_failure_updated)

print(f"Original failure samples (target=1): {len(lag_failure_updated)}")
print(f"New non-failure samples (target=0): {len(non_failure_df)}")
print(f"Total balanced dataset: {len(balanced_dataset)}")
print(f"Dataset has {len(balanced_dataset.columns)} features")
print(f"\nTarget distribution:")
print(balanced_dataset['target'].value_counts())

# Verify all your features are present
print(f"\nFeatures included: {list(balanced_dataset.columns)}")

# Check for any missing values
print(f"\nMissing values per column:")
print(balanced_dataset.isnull().sum().sum())

Original failure samples (target=1): 11
New non-failure samples (target=0): 11
Total balanced dataset: 22
Dataset has 65 features

Target distribution:
0        11
comp4     7
comp2     3
comp1     1
Name: target, dtype: int64

Features included: ['datetime', 'volt', 'rotate', 'pressure', 'vibration', 'errorID', 'comp', 'failure', 'target', 'volt_lag_1h', 'volt_lag_6h', 'volt_lag_12h', 'volt_lag_24h', 'volt_mean_24h', 'volt_std_24h', 'volt_min_24h', 'volt_max_24h', 'volt_mean_6h', 'volt_std_6h', 'rotate_lag_1h', 'rotate_lag_6h', 'rotate_lag_12h', 'rotate_lag_24h', 'rotate_mean_24h', 'rotate_std_24h', 'rotate_min_24h', 'rotate_max_24h', 'rotate_mean_6h', 'rotate_std_6h', 'pressure_lag_1h', 'pressure_lag_6h', 'pressure_lag_12h', 'pressure_lag_24h', 'pressure_mean_24h', 'pressure_std_24h', 'pressure_min_24h', 'pressure_max_24h', 'pressure_mean_6h', 'pressure_std_6h', 'vibration_lag_1h', 'vibration_lag_6h', 'vibration_lag_12h', 'vibration_lag_24h', 'vibration_mean_24h', 'vibration_std_24h'

In [40]:
display(machine_failure)

display(balanced_dataset)

,datetime,machineID,volt,rotate,pressure,vibration,model,age,errorID,comp,failure
96,2015-01-05 06:00:00,1,179.303153,499.777962,111.833028,52.383097,model3,18,0,comp4,comp4
97,2015-01-05 06:00:00,1,179.303153,499.777962,111.833028,52.383097,model3,18,0,comp1,comp4
1539,2015-03-06 06:00:00,1,198.257975,456.862342,89.333995,38.671900,model3,18,0,comp1,comp1
2620,2015-04-20 06:00:00,1,180.050801,346.362480,105.661164,39.218055,model3,18,0,comp2,comp2
4061,2015-06-19 06:00:00,1,187.673963,493.005160,105.334392,53.963961,model3,18,0,comp1,comp4
4062,2015-06-19 06:00:00,1,187.673963,493.005160,105.334392,53.963961,model3,18,0,comp4,comp4
5862,2015-09-02 06:00:00,1,144.094532,409.380150,106.720871,57.454990,model3,18,0,comp1,comp4
5863,2015-09-02 06:00:00,1,144.094532,409.380150,106.720871,57.454990,model3,18,0,comp4,comp4
6945,2015-10-17 06:00:00,1,178.322428,383.715256,79.704008,43.213417,model3,18,0,comp4,comp2
6946,2015-10-17 06:00:00,1,178.322428,383.715256,79.704008,43.213417,model3,18,0,comp2,comp2


,datetime,volt,rotate,pressure,vibration,errorID,comp,failure,target,volt_lag_1h,...,error_count_6h,error_count_24h,maint_count_6h,maint_count_24h,hour,day_of_week,is_weekend,is_working_hours,hours_since_maint,hours_since_error
0,2015-01-04 06:00:00,0.386535,0.553421,0.403796,0.568691,5,0,0,comp4,0.433966,...,5.0,9.0,0.0,0.0,6,6,1,0,72,0
1,2015-01-04 06:00:00,0.386535,0.553421,0.403796,0.568691,5,0,0,comp4,0.433966,...,5.0,9.0,0.0,0.0,6,6,1,0,72,0
2,2015-01-29 17:00:00,0.491148,0.567421,0.605486,0.417514,0,0,0,0,0.229646,...,0.0,0.0,0.0,0.0,17,3,0,1,227,61
3,2015-02-01 11:00:00,0.197684,0.468569,0.332441,0.227992,0,0,0,0,0.541692,...,0.0,0.0,0.0,0.0,11,6,1,1,293,127
4,2015-03-05 06:00:00,0.754059,0.400178,0.465682,0.442350,1,0,0,comp1,0.606900,...,1.0,1.0,0.0,0.0,6,3,0,0,336,0
5,2015-03-13 21:00:00,0.464008,0.491742,0.326767,0.435240,0,0,0,0,0.338001,...,0.0,0.0,0.0,0.0,21,4,0,0,183,207
6,2015-04-19 06:00:00,0.317181,0.392709,0.369041,0.328448,2,0,0,comp2,0.476150,...,2.0,2.0,0.0,0.0,6,6,1,0,336,0
7,2015-05-22 18:00:00,0.549801,0.578536,0.773275,0.430296,0,0,0,0,0.613994,...,0.0,0.0,0.0,0.0,18,4,0,0,60,35
8,2015-05-23 03:00:00,0.388520,0.663687,0.721823,0.288828,0,0,0,0,0.520877,...,0.0,0.0,0.0,0.0,3,5,1,0,69,44
9,2015-06-14 04:00:00,0.519190,0.477925,0.413712,0.346261,0,0,0,0,0.452366,...,0.0,0.0,0.0,0.0,4,6,1,0,238,118


<br> <br> <br>

---

<br> <br>

# Safe Zone Sampling Methodology for Balanced Dataset Creation in Predictive Maintenance

## Abstract

This methodology presents a novel approach for creating balanced datasets in predictive maintenance applications, specifically addressing the challenge of generating representative non-failure samples while avoiding temporal contamination. The proposed Safe Zone Sampling (SZS) methodology ensures temporal integrity by establishing exclusion zones around failure events, thereby preventing data leakage and maintaining the predictive validity of machine learning models.

## Introduction

In predictive maintenance applications, the creation of balanced datasets presents unique challenges due to the temporal nature of machinery degradation and the need to predict failures within specific time horizons. Traditional random sampling approaches for generating non-failure instances may inadvertently include data points from periods of incipient failure, leading to compromised model performance and unrealistic predictive capabilities. This methodology addresses these challenges through a systematic approach to temporal data sampling that maintains the integrity of predictive modeling objectives.

## Methodology

### 2.1 Problem Formulation

Given a time series dataset of machine operational parameters, the objective is to create a balanced binary classification dataset where:
- Positive samples (target = 1) represent operational states exactly τ hours before actual failure events
- Negative samples (target = 0) represent stable operational states with no impending failures within a defined temporal horizon

The critical challenge lies in ensuring that negative samples are selected from periods that are genuinely representative of stable operation, avoiding the temporal vicinity of failure events where degradation processes may already be initiated.

### 2.2 Safe Zone Sampling Framework

The Safe Zone Sampling methodology operates on the principle of temporal exclusion zones, defined as time intervals around failure events where data sampling is prohibited. The framework consists of four primary components:

#### 2.2.1 Failure Event Identification
Let F = {f₁, f₂, ..., fₙ} represent the set of failure timestamps identified in the historical dataset. Each failure event fᵢ corresponds to a recorded machinery failure with timestamp tᵢ.

#### 2.2.2 Exclusion Zone Definition
For each failure event fᵢ at time tᵢ, an exclusion zone Eᵢ is defined as:

Eᵢ = [tᵢ - (τ + β), tᵢ]

where:
- τ represents the prediction horizon (typically 24 hours)
- β represents the safety buffer (recommended 48 hours)
- The total exclusion period extends (τ + β) hours before the failure event

The safety buffer β serves to account for potential degradation patterns that may commence before the explicit prediction horizon, ensuring that sampled non-failure instances represent genuinely stable operational periods.

#### 2.2.3 Safe Zone Identification
The complement of all exclusion zones defines the safe sampling space S:

S = T \ ⋃ᵢ₌₁ⁿ Eᵢ

where T represents the complete temporal domain of the dataset. Safe zones correspond to time periods where machinery operation is sufficiently distant from any failure events, both preceding and following.

#### 2.2.4 Stratified Sampling Strategy
To ensure representativeness across different operational conditions, a stratified sampling approach is employed within the safe zones. Stratification variables may include:
- Machine component identifiers
- Temporal factors (working hours, weekends, seasonal patterns)
- Operational modes or states

The stratified sampling ensures that non-failure samples maintain proportional representation across different operational scenarios present in the failure dataset.

### 2.3 Implementation Algorithm

The Safe Zone Sampling algorithm proceeds as follows:

1. **Temporal Preprocessing**: Convert all timestamps to a standardized datetime format and sort chronologically.

2. **Exclusion Zone Construction**: For each failure timestamp tᵢ, compute the exclusion interval [tᵢ - (τ + β), tᵢ].

3. **Safe Candidate Identification**: Filter the complete dataset to retain only observations falling outside all exclusion zones and satisfying the condition failure = 0.

4. **Quality Assurance**: Remove any timestamps already included in the positive sample set to prevent duplication.

5. **Stratified Sampling**: Apply stratified random sampling within safe zones to select n non-failure samples, where n typically equals the number of positive samples to achieve class balance.

6. **Feature Alignment**: Ensure consistent feature representation between positive and negative samples by maintaining identical feature sets and handling any missing values appropriately.

### 2.4 Advantages and Theoretical Justification

The Safe Zone Sampling methodology provides several theoretical and practical advantages:

#### 2.4.1 Temporal Integrity
By establishing exclusion zones, the methodology prevents temporal data leakage that could artificially inflate model performance during validation. This ensures that predictive models are evaluated under realistic conditions that mirror actual deployment scenarios.

#### 2.4.2 Degradation Process Consideration
The safety buffer β accounts for the fact that machinery degradation is typically a gradual process that may commence well before observable failure symptoms. Traditional sampling approaches may inadvertently include early degradation stages as negative examples, leading to model confusion and reduced predictive accuracy.

#### 2.4.3 Statistical Validity
The stratified sampling component ensures that non-failure samples maintain statistical representativeness across different operational conditions, preventing bias toward specific operational scenarios and improving model generalizability.

#### 2.4.4 Scalability and Adaptability
The methodology is parametrically configurable, allowing adjustment of the prediction horizon τ and safety buffer β based on domain expertise, machinery characteristics, and specific application requirements.

## Experimental Validation Considerations

When implementing this methodology, several validation considerations should be addressed:

### 3.1 Temporal Cross-Validation
Standard k-fold cross-validation is inappropriate for temporal data due to potential information leakage. Time-series cross-validation techniques should be employed, ensuring that training data always precedes validation data chronologically.

### 3.2 Safety Buffer Sensitivity Analysis
The selection of the safety buffer parameter β should be validated through sensitivity analysis, examining model performance across different buffer values to identify optimal settings for specific applications.

### 3.3 Class Balance Evaluation
While achieving numerical class balance is important, the quality and representativeness of sampled instances should be evaluated to ensure that the resulting dataset accurately reflects real-world operational conditions.

## Limitations and Future Work

The Safe Zone Sampling methodology assumes that failure events are sufficiently sparse to allow adequate safe zone identification. In cases of frequent failures or cascading failure scenarios, the available safe sampling space may be limited. Future work should address adaptive sampling strategies for high-failure-rate systems and investigate the integration of domain-specific knowledge for refined exclusion zone definition.

## Conclusion

The Safe Zone Sampling methodology provides a principled approach to balanced dataset creation in predictive maintenance applications. By incorporating temporal exclusion zones and stratified sampling strategies, the methodology addresses critical challenges in time-series classification while maintaining the statistical integrity required for robust machine learning model development. The parametric nature of the approach allows for customization across different industrial applications and machinery types, making it a versatile tool for predictive maintenance research and implementation.